Return values define what data a function sends back to the caller when it finishes executing,

while scope determines where in your code a varaible can be accessed or modified

## Return values and Early exit guard clauses

The return statement immediately stops function execution and passes a value back to the caller.

If a function reaches the end without hitting return (or uses a bare return), Python automatically return None.

- Returning Multiple Values: seperating values with commas pack them into a tuple, which the caller can unpack directly.
- Guard Clauses (Early Returns): Instead of nesting deep if/else blocks, backend developers check for invalid states first and return early.

In [2]:
# Returning multiple values as a tuple

def calculate_pagination(total_items: int, page_size: int) -> tuple[int,int]:
    total_pages = (total_items * page_size - 1)// page_size
    remainder = total_items % page_size
    return total_pages, remainder

pages, last_page_count = calculate_pagination(53,20)
print(f"Total pages: {pages}, Last page items: {last_page_count}")

Total pages: 52, Last page items: 13


In [4]:
# Guard clause pattern (avoids deep nesting)
def get_discount_rate(user_role:str , is_active: bool) -> float:
    if not is_active:
        return 0.0
    if user_role == "admin":
        return 0.25
    if user_role == "intern":
        return 0.15
    return 0.5

print(get_discount_rate("intern", True))

0.15


# Variable Scope (The LEGB rule)

Python resolves variable names by searching four layers from innermost to outermost (LEGB)

- L -Local: variables created inside the current function. They are destroyed as soon as the function returns
- E -Enclosing : Variables in outer functions (used in nested functions and decorators)
- G - Global : Variables defind at the top level of the py file
- B - Built-in : Names preloaded by Python (len,print,int,..)




In [6]:
API_VERSION = "v1" # Global scope (readable everywhere)

def build_endpoint(resource: str) -> str:
    # 'full_path' exists ONLY in Local scope inside build_endpoint()
    full_path = f"/api/{API_VERSION}/{resource}"
    return full_path

print(build_endpoint("users")) #will get output
#print(full_path) # Raises NameError: name "full_path" is not defined since it is a local variable

/api/v1/users


## Avoiding Global state mutation

Functions can read global variables freely, but assigning to a variable inside a function creates a brand-new local variable unless we explicitly use the 'global' keyword


In production APIs, never mutate global variables inside functions - multiple concurrent requests will overwrite each other's data.

Always pass data in via parametrs and send updates out via return.

In [11]:
request_count = 0

def increment_wrong() -> None:
    # UnboundLocalError : Python sees asssignment and treats request_count as local
    request_count += 2
    pass

# Clean, stateless approach: accept current value, return new value

def increment_clean(current_count: int) -> int:
    return current_count + 1

request_count = increment_clean(request_count)

print("Updated count:", request_count)
req_wcount = increment_wrong()
print(request_count)
print(req_wcount)

Updated count: 1


UnboundLocalError: cannot access local variable 'request_count' where it is not associated with a value

## Realworld Backend and API examples

In FastAPI applications, each incoming HTTP request runs inside its own local function scope so user sessions never bleed into one another

In [14]:
# Example : Request-Scoped Token Validation Service

SECRET_PREFIX = "sk_live_" #Globalconstant (safe because it is never mutated)

def verify_api_token(auth_header: str | None) -> tuple[bool,str]:
    """ validates an Authorization header and returns (is_valid, message_or_token)"""

    if not auth_header:
        return False, "Missing Authorization header"
    if not auth_header.startswith("Bearer "):
        return False, "Invalid scheme: expected Bearer token"
    
    # Local vairable isolated into this specific request
    token = auth_header.removeprefix("Bearer ").strip()

    if not token.startswith(SECRET_PREFIX):
        return False, "Invalid API key prefix"
    
    return True, token

is_valid, result = verify_api_token("Bearer sk_live_00845a")
print(f"Valid: {is_valid} | Payload: {result}")

Valid: True | Payload: sk_live_00845a
